# Greer downloader

The general idea is to remake the Mattress factory site with everything dumped into one place.

Underneath, the site is using CollectiveAccess

In [ ]:
from selenium import webdriver
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys

# 1. Configure Firefox Options (Optional)
options = Options()
# To run without opening a GUI window, uncomment the line below:
# options.add_argument("-headless") 

# 2. Initialize the Driver (GeckoDriver is handled automatically)
driver = webdriver.Firefox(options=options)

driver.get(f"https://lankton.mattress.org/")

In [ ]:
import time
import re
import requests
import io
from PIL import Image

url = "https://example.com"

# Stream the response so it doesn't download the body automatically
response = requests.get(url, stream=True)

searches = [
    # '1950s',
    # '1960s',
    # '1970s',
    # '1980s',
    # '1990s',
    # 'greer',
    '*' # This seems to be everything
]

rows = []

for search_term in searches:
    driver.get(f"https://lankton.mattress.org/Search/objects/search/{search_term}")
    time.sleep(1)
    
    body = driver.find_element(By.TAG_NAME, "body")

    h2_elements = driver.find_elements(By.TAG_NAME, "h2")
    count = None
    
    for element in h2_elements:
        match = re.search(r"\b([\d,]+)\s+Object\s+Results", element.text)
        if match:
            count = int(match.group(1).replace(",", ""))
    if count is None:
        count = 14282
    print(f"{search_term}: {count}")
    
    results = []
    pressed_end = 3
    last_n = 0
    while no_of_pagedowns:
        results = driver.find_elements(By.CSS_SELECTOR, ".bResultItemContent")
        if len(results) >= count:
            break
        is_loading = len(driver.find_elements(By.CLASS_NAME, "jscroll-loading"))
        if not is_loading and pressed_end:
            body.send_keys(Keys.END)
            pressed_end -= 1
        elif last_n != len(results):
            last_n = len(results)
            pressed_end = 3
            print(f"{len(results)}/{count}, {pressed_end - 3}/3", end="\r")
        if not pressed_end:
            break
        time.sleep(0.5)
    print()
    print(f"Done searching {search_term}")
    for result in results:
        archive_id = result.find_element(By.CSS_SELECTOR, ".bResultItemText small").text
        link = result.find_element(By.CSS_SELECTOR, ".bResultItemImg a").get_attribute("href")
        thumb = result.find_element(By.CSS_SELECTOR, ".bResultItemImg img").get_attribute("src")
        alt = result.find_element(By.CSS_SELECTOR,  ".bResultItemImg img").get_attribute("alt")
        rows.append([archive_id, link, thumb, alt])
print()
print("Done")

In [ ]:
from copy import deepcopy
rows_bak_bak = deepcopy(rows_bak)
rows_bak = deepcopy(rows)

In [ ]:
import pandas as pd
df = pd.DataFrame(rows, columns=['archive_id', 'link', 'thumb', 'alt']).drop_duplicates(subset=['archive_id'])
df = df.set_index(['archive_id'])
df.sort_index(inplace=True)
print(len(df), len(rows))
df.to_csv("greer_archive.csv.gz", index=True, compression="gzip")